# 4a. Forced response figures: the mean and signal to noise

The figures for the forced response, drawn from what notebook 03 saved, so nothing slow runs here. Notebook 03 explains each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebook 03:

| Notebook | Sections |
|---|---|
| 04a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| 04b | 3. Quantile response; 4. Change in width, and which tail |
| 04c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| 04d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 1. Mean change | final 21 years against hist-nat over the same years | hist-nat bootstrap |
| 2. Signal to noise | \|S/N\|: how far apart the two distributions have moved | none: a measure, not a test |

**By SAM phase.** Section 1 ends with the result in the seasons of each phase of the Southern Annular Mode alone (negative, neutral and positive, cut at hist-nat's terciles of the index: notebook 03, *The SAM phases*), the phases as rows.

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The black lines on the maps of section 1 are the sea-ice edge (the 15% contour of the ensemble-mean concentration over the final 21 years, from notebook 09): hist-nat's dashed, the experiment's solid; the multi-model figures draw the edge of the models' median concentration. Without the sea ice they are left out. In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook 06. The cells before each figure pick out its data and show them.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook 03. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from extant import config, paths, sam, sea_ice
from extant import significance as sig, storage
from extant.logger import setup_logging
from extant.plots import forced_response as fr, paper_panels, sea_ice as sip

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table

In [ ]:
%matplotlib inline

## Settings

In [ ]:
VARIABLE = 'tas'

WINDOW = config.WINDOW
LAST_FULL_YEAR = config.LAST_FULL_YEAR

#(t): The experiments, in the order of every figure, and the seasons, the columns of every map
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the "what goes in" and "one model" figures
MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']

#(t): Significance level of every test
ALPHA = 0.05

#(t): The |S/N| levels section 2 counts the models above
SN_LEVELS = [0.5, 1, 2]

#(t): The SAM phases of 1.5, in the order of their rows, and how the rows are labelled
PHASES = list(sam.PHASES)
PHASE_LABELS = {phase: f'SAM\n{phase}' for phase in PHASES}

#(t): The share of the models that must have a significant change of one sign for a multi-model change to be robust
ROBUST = config.SIGNIFICANT_THRESHOLD


def result_path(name):
    """Where notebook 03 saved a result: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks 05 and 06), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook 06 (*Check the colour ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE, alpha=ALPHA)

In [ ]:
#(t): The sea-ice zones (notebook 09a, section 7), coloured as in notebook 09b
zone_scale = fr.class_scale(sea_ice.ZONES, ['#f4f3ef', '#2c7fb8', '#e6550d', '#31a354', '#bdbdbd'], 'Zone')

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the saved results

Run **one** of the next two sections. Both leave `RUN`, which says which saved results to open.

### Sample data

In [ ]:
#(t): The results notebook 03 saved from the sample
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
#(t): The results notebook 03 saved from the full data
RUN = 'full'

### The saved results

Everything sections 1 and 2 use, and the check of the results below, from `results/<variable>/<RUN>/` under `paths.DATA_DIR` or `paths.SCRATCH` (the log gives each full path), loaded into memory. Grouped by the section that uses them.

In [ ]:
#(t): Section 1: the hist-nat bootstrap of the mean change, with hist-nat's and every experiment's final-years mean
mean_bootstrap_ds = storage.open_dataset(result_path('mean_bootstrap'), load=True)

#(t): Section 2: the forced responses, each experiment's noise, and the signal, noise and S/N made from them
lesfmip_forced_ds = storage.open_dataset(result_path('forced'), load=True)
lesfmip_experiment_noise_ds = storage.open_dataset(result_path('experiment_noise'), load=True)
lesfmip_signal_ds = storage.open_dataset(result_path('signal'), load=True)
lesfmip_noise_ds = storage.open_dataset(result_path('noise'), load=True)
lesfmip_seasonal_sn_ds = storage.open_dataset(result_path('sn'), load=True)

#(t): The check of the results: the quantile response and the width test (their figures are in notebook 04b)
quantile_response_da = storage.open_dataarray(result_path('quantile_response'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)

#(t): By SAM phase (1.5): how often each experiment is in each phase, and the mean change in each phase
sam_phase_share_da = storage.open_dataarray(result_path('sam_phase_share'), load=True)
sam_mean_bootstrap_ds = storage.open_dataset(result_path('sam_mean_bootstrap'), load=True)

In [ ]:
#(t): What each result holds: its dims and its variables
results = {
    'mean_bootstrap_ds': mean_bootstrap_ds, 'lesfmip_forced_ds': lesfmip_forced_ds,
    'lesfmip_experiment_noise_ds': lesfmip_experiment_noise_ds, 'lesfmip_signal_ds': lesfmip_signal_ds,
    'lesfmip_noise_ds': lesfmip_noise_ds, 'lesfmip_seasonal_sn_ds': lesfmip_seasonal_sn_ds,
    'quantile_response_da': quantile_response_da, 'qrange_change_ds': qrange_change_ds,
    'sam_phase_share_da': sam_phase_share_da, 'sam_mean_bootstrap_ds': sam_mean_bootstrap_ds,
}
pd.DataFrame({
    name: {'dims': dict(result.sizes), 'variables': ', '.join(result.data_vars) if isinstance(result, xr.Dataset) else ''}
    for name, result in results.items()
}).T

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = mean_bootstrap_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the maps of section 1, and the sea-ice results of 1.6, come from notebook 09a. Run **one** of the next two sections. Both leave `ice_climatology_da` (the ensemble-mean concentration over the final years), and `model_ice_da` and `median_ice_da` (`MODEL`'s concentration, and the models' median, for the edges of the one-model and multi-model maps); without sea ice they are `None` and the figures leave the edge out. *With sea ice* also opens the zones and the changes 1.6 uses; without it, 1.6 cannot run.

#### Without sea ice

Before notebook 09 has saved the sea ice (on the sample, that is when the notebooks are run in order).

In [ ]:
ice_climatology_da = model_ice_da = median_ice_da = None

#### With sea ice

In [ ]:
#(t): Where notebook 09a saved its results, for the same RUN: JASMIN's scratch
sea_ice_results_dir = Path('/work/scratch-pw4/aborow/ExtAnt') / 'results' / 'siconc' / RUN
sea_ice_results_dir

In [ ]:
#(t): The ensemble-mean concentration over the final years (%), MODEL's, and the models' median (for the edges)
ice_climatology_da = xr.open_zarr(sea_ice_results_dir / 'climatology.zarr')['siconc'].load()
model_ice_da = ice_climatology_da.sel(model=MODEL)
median_ice_da = ice_climatology_da.median('model')

In [ ]:
#(t): Each cell's zone in the final years (the numbers of sea_ice.ZONES), every model, experiment and season
ice_zones_da = xr.open_zarr(sea_ice_results_dir / 'zones.zarr')['zone'].load()
ice_zones_da

In [ ]:
#(t): The temperature changes averaged over each zone, and each zone's area (million km²)
tas_zone_means_ds = xr.open_zarr(sea_ice_results_dir / 'zone_means.zarr').load()
tas_zone_means_ds

In [ ]:
#(t): The change in extent (million km²) and in the edge's latitude (degrees) against hist-nat
extent_change_ds = xr.open_zarr(sea_ice_results_dir / 'extent_change.zarr').load()
extent_change_ds

In [ ]:
#(t): The temperature changes averaged over the cells with ice in hist-nat's final years
ice_region_change_ds = xr.open_zarr(sea_ice_results_dir / 'ice_region_change.zarr').load()
ice_region_change_ds

## Check the results

The headline changes of every model, experiment and season, averaged over the whole area (cos-latitude weighted). A model whose numbers sit far from the others' has probably gone wrong before notebook 03 (in its raw files or their conversion), and every figure below would show it; look at its members in notebook 03 before reading anything into it.

In [ ]:
#(t): The headline changes side by side: the mean, three quantiles, the width, and S/N in the centre of the final years
headline_ds = xr.Dataset({
    'mean change': mean_bootstrap_ds['mean_change'],
    'ΔQ05': quantile_response_da.sel(quantile=0.05, drop=True),
    'ΔQ50': quantile_response_da.sel(quantile=0.50, drop=True),
    'ΔQ95': quantile_response_da.sel(quantile=0.95, drop=True),
    'Δ(Q95 − Q05)': qrange_change_ds['qrange_change'],
    'S/N': lesfmip_seasonal_sn_ds[VARIABLE].isel(year=LAST_FULL_YEAR, drop=True),
})
headline_ds = headline_ds.sel(experiment=EXPERIMENTS, season=SEASONS)
headline_ds

In [ ]:
#(t): Their area means, one row per model, experiment and season
headline_area_ds = sig.area_mean(headline_ds)
headline_area_df = headline_area_ds.to_dataframe().round(2)
headline_area_df

## 1. Mean change

The mean of each experiment's final 21 years minus hist-nat's over the same years, members and years pooled (notebook 03, section 4):

$$
\Delta\bar{x} = \bar{x}_{\mathrm{exp}} - \bar{x}_{\mathrm{nat}}.
$$

It is tested with the hist-nat bootstrap: the experiment's mean against the means of 1000 trials, each 10 random hist-nat members over the same 21 years, so natural variability, the correlation between a member's years included, sets what counts as significant. The change is significant where $p < 0.05$. (Notebook 03 also saves the two Welch t-tests, `ttest` and `ttest_rolling_median`, which count correlated values as independent and so call more changes significant, and the member-block permutation test, `mean_block`.)

### 1.1 What goes in

hist-nat's mean and `EXPERIMENT`'s, both over the final 21 years, for `MODEL`; the change between them; and the test's p-value.

In [ ]:
#(t): hist-nat's mean over the final WINDOW years, MODEL, every season
hist_nat_mean_da = mean_bootstrap_ds['hist_nat_mean'].sel(model=MODEL, season=SEASONS)
hist_nat_mean_da

In [ ]:
#(t): EXPERIMENT's over the same years
experiment_mean_da = mean_bootstrap_ds['experiment_mean'].sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
experiment_mean_da

In [ ]:
#(t): The change (mean_bootstrap_ds['mean_change'] holds the same)
mean_change_da = experiment_mean_da - hist_nat_mean_da
mean_change_da

In [ ]:
#(t): Its p-value, from the hist-nat bootstrap
mean_pvalue_da = mean_bootstrap_ds['pvalue'].sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
mean_pvalue_da

In [ ]:
#(t): Not significant: p >= ALPHA (False where there is no data)
mean_not_significant_da = mean_pvalue_da >= ALPHA

In [ ]:
panels = fr.row_grid(
    [fr.Row(hist_nat_mean_da, scales.temperature, f'hist-nat\nfinal {WINDOW} years'),
     fr.Row(experiment_mean_da, scales.temperature, f'{EXPERIMENT}\nfinal {WINDOW} years'),
     fr.Row(mean_change_da, scales.change, f'Change\n{EXPERIMENT} − hist-nat', not_significant=mean_not_significant_da,
            edges=sip.edge_pair(model_ice_da, EXPERIMENT)),
     fr.Row(mean_pvalue_da, scales.pvalue, 'p-value\nhist-nat bootstrap')],
    title=f'What goes into the mean change: {MODEL}, {EXPERIMENT}',
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

### 1.2 One model

The change for every experiment of `MODEL`, dotted where it is not significant (1.1 shows the p-values).

In [ ]:
#(t): MODEL's change in every experiment and season
model_mean_change_da = mean_bootstrap_ds['mean_change'].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_mean_change_da

In [ ]:
#(t): Its p-value
model_mean_pvalue_da = mean_bootstrap_ds['pvalue'].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_mean_pvalue_da

In [ ]:
#(t): Not significant
model_mean_not_significant_da = model_mean_pvalue_da >= ALPHA

In [ ]:
panels = fr.map_grid(
    model_mean_change_da, scales.change,
    title=f'{MODEL}: mean change, final {WINDOW} years against hist-nat',
    row_dim='experiment', col_dim='season',
    not_significant=model_mean_not_significant_da,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
    edges=sip.edge_pair(model_ice_da, EXPERIMENTS),
)

### 1.3 Every model

One figure per experiment: every model down, every season across.

In [ ]:
#(t): The change for every model, experiment and season
all_mean_change_da = mean_bootstrap_ds['mean_change'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_mean_change_da

In [ ]:
#(t): Significant: p < ALPHA (False where there is no data)
all_mean_significant_da = mean_bootstrap_ds['pvalue'].sel(experiment=EXPERIMENTS, season=SEASONS) < ALPHA
all_mean_significant_da

In [ ]:
#(t): Not significant, where there is data
all_mean_not_significant_da = ~all_mean_significant_da & all_mean_change_da.notnull()

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_mean_change_da.sel(experiment=experiment), scales.change, col_dim='season',
        title=f'Mean change, final {WINDOW} years against hist-nat, every model: {experiment}',
        not_significant=all_mean_not_significant_da.sel(experiment=experiment),
        not_significant_label=f'Not significant (hist-nat bootstrap, p ≥ {ALPHA:g})',
        edges=sip.edge_pair(ice_climatology_da, experiment),
    )

### 1.4 How many models

The multi-model median of the change (top row), then at each grid point the number of models whose mean change is significant and positive, and significant and negative. One figure per experiment.

In [ ]:
#(t): The multi-model median of the change (a median, so one model's bad data cannot drag it)
median_mean_change_da = all_mean_change_da.median('model')
median_mean_change_da

In [ ]:
#(t): How many models have a significant increase, and how many a significant decrease
mean_counts_da = sig.significant_counts(all_mean_change_da, all_mean_significant_da)
mean_counts_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.count_grid(
        mean_counts_da.sel(experiment=experiment), N_MODELS,
        above=[fr.Row(median_mean_change_da.sel(experiment=experiment), scales.change, 'Multi-model\nmedian',
                      edges=sip.edge_pair(median_ice_da, experiment))],
        title=f'How many models have a significant change in the mean: {experiment}',
        label=f'Number of models with a significant decrease or increase, p < {ALPHA:g}',
        col_dim='season',
        row_labels={'increase': 'Significant\nincrease', 'decrease': 'Significant\ndecrease'},
        edges=sip.edge_pair(median_ice_da, experiment),
    )

### 1.5 By SAM phase

The mean change in the seasons of each SAM phase alone (notebook 03, *The SAM phases* and section 4, *By SAM phase*): each experiment's seasons in a phase against hist-nat's in the same phase, the phases cut at hist-nat's terciles of the SAM index (so a third of hist-nat's seasons fall in each). First how often each experiment is in each phase; then the change in each phase for `MODEL`, dotted where it is not significant; the multi-model median, dotted where it is not robust (fewer than two-thirds of the models significant with one sign); and how many models are significant with each sign.

In [ ]:
#(t): How often each experiment of MODEL is in each phase over the final WINDOW years
model_phase_share_da = sam_phase_share_da.sel(model=MODEL, season=SEASONS)
model_phase_share_da

In [ ]:
model_phase_share_da.to_series().unstack('season')[SEASONS].round(2)

In [ ]:
#(t): The share of the seasons in the positive phase, every model and experiment
sam_phase_share_da.sel(phase='positive', season=SEASONS).to_series().unstack('season')[SEASONS].round(2)

In [ ]:
#(t): MODEL's change in the mean in each phase, every experiment and season
model_phase_mean_change_da = sam_mean_bootstrap_ds['mean_change'].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_phase_mean_change_da

In [ ]:
#(t): Not significant: p >= ALPHA (False where there is no data)
model_phase_mean_not_significant_da = sam_mean_bootstrap_ds['pvalue'].sel(model=MODEL, experiment=EXPERIMENTS,
                                                                          season=SEASONS) >= ALPHA

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        model_phase_mean_change_da.sel(experiment=experiment), scales.change,
        title=f'{MODEL}: mean change in each SAM phase, final {WINDOW} years against hist-nat: {experiment}',
        row_dim='phase', col_dim='season', row_labels=PHASE_LABELS,
        not_significant=model_phase_mean_not_significant_da.sel(experiment=experiment),
        not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
    )

In [ ]:
#(t): The change in each phase, every model, experiment and season
all_phase_mean_change_da = sam_mean_bootstrap_ds['mean_change'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_phase_mean_change_da

In [ ]:
#(t): Significant: p < ALPHA (False where there is no data)
all_phase_mean_significant_da = sam_mean_bootstrap_ds['pvalue'].sel(experiment=EXPERIMENTS, season=SEASONS) < ALPHA

In [ ]:
#(t): The multi-model median
median_phase_mean_change_da = all_phase_mean_change_da.median('model')
median_phase_mean_change_da

In [ ]:
#(t): Where the models agree: +1 (-1) where at least ROBUST of them have a significant increase (decrease)
phase_mean_robust_da = sig.robust_sign(all_phase_mean_change_da, all_phase_mean_significant_da, threshold=ROBUST)
phase_mean_robust_da

In [ ]:
#(t): Not robust, where there is data
phase_mean_not_robust_da = phase_mean_robust_da == 0

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        median_phase_mean_change_da.sel(experiment=experiment), scales.change,
        title=f'Mean change in each SAM phase, multi-model median: {experiment}',
        row_dim='phase', col_dim='season', row_labels=PHASE_LABELS,
        not_significant=phase_mean_not_robust_da.sel(experiment=experiment),
        not_significant_label='Not robust (fewer than two-thirds of the models significant with one sign)',
    )

In [ ]:
#(t): How many models have a significant increase, and how many a significant decrease, in each phase
phase_mean_counts_da = sig.significant_counts(all_phase_mean_change_da, all_phase_mean_significant_da)
phase_mean_counts_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.count_grid(
        phase_mean_counts_da.sel(experiment=experiment), N_MODELS,
        title=f'How many models have a significant change in the mean in each SAM phase: {experiment}',
        label=f'Number of models with a significant decrease or increase, p < {ALPHA:g}',
        col_dim='season', group_dim='phase',
        row_labels={'increase': 'Significant\nincrease', 'decrease': 'Significant\ndecrease'},
    )

### 1.6 Sea ice and the mean change

Where the ice changed, and how the mean changed there (notebook 09a, section 7). Each ocean cell is sorted by its concentration in hist-nat's final years and the experiment's: open water in both, ice in both, ice lost, ice gained, or the continent. If losing the ice drives the warming, "ice lost" warms more than "open water" and "ice in both", most in autumn and winter. First what goes in for `MODEL` and `EXPERIMENT`; then the mean change averaged over each zone, every model; then, one point per model and forcing, the change in extent against the mean change over hist-nat's ice (where ice can be lost). Needs the sea ice (*With sea ice* above).

In [ ]:
#(t): MODEL's zones for EXPERIMENT, every season, and its two edges: hist-nat's (dashed) and EXPERIMENT's (solid)
model_zones_da = ice_zones_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
model_zone_edges_da = sip.edge_pair(model_ice_da.sel(season=SEASONS), EXPERIMENT)
model_zones_da

In [ ]:
#(t): What goes in: the zones, and MODEL's mean change for EXPERIMENT, with both edges
panels = fr.row_grid(
    [fr.Row(model_zones_da, zone_scale, 'Zone', edges=model_zone_edges_da),
     fr.Row(mean_bootstrap_ds['mean_change'].sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), scales.change,
            f'Δ mean\n{EXPERIMENT} − hist-nat', edges=model_zone_edges_da)],
    title=f'Where the ice changed, and the mean: {MODEL}, {EXPERIMENT}, final {WINDOW} years',
)

In [ ]:
#(t): The mean change averaged over each zone, every model, experiment and season (°C)
zone_mean_change_da = tas_zone_means_ds['mean']
zone_mean_change_da

In [ ]:
fig, axes = sip.zone_changes(zone_mean_change_da, 'Δ mean (°C)', experiments=EXPERIMENTS, seasons=SEASONS,
                             title=f'Change in the mean temperature in each zone, final {WINDOW} years against hist-nat')

In [ ]:
#(t): The change in extent (million km²) and the mean change over hist-nat's ice (°C), every model and experiment
extent_change_da = extent_change_ds['extent_change']
ice_region_mean_change_da = ice_region_change_ds['mean']
ice_region_mean_change_da

In [ ]:
panels = paper_panels.scatter_by_season(extent_change_da, ice_region_mean_change_da, 'Δ extent (million km²)',
                                        "Δ mean over hist-nat's ice (°C)", experiments=EXPERIMENTS, columns=SEASONS,
                                        style=fr.NOTEBOOK)

## 2. Signal to noise

S/N in the centre of the final 21 years (2003), the last year with a complete noise window, so it covers the same years as the tests (notebook 03, section 5):

- the **signal** $S$ is the experiment's forced response minus hist-nat's, where a forced response is the ensemble mean smoothed with LOWESS (81 years);
- the **noise** $N$ pools the two experiments' standard deviations $\sigma$ of each member about its forced response, over the 21 years centred on that year: $N = \sqrt{\tfrac{1}{2}\left(\sigma^2_{\mathrm{exp}} + \sigma^2_{\mathrm{nat}}\right)}$.

S/N is not a test here. It says how far apart the experiment's and hist-nat's distributions have moved, in units of their year-to-year noise: two normal distributions 0.5, 1 and 2 noise apart share about 80%, 62% and 32% of their area. The maps show $|\mathrm{S/N}|$, whichever way the shift goes (the signal gives its sign): yellow below 1, orange from 1 to 2, red above 2. Section 2.4 counts the models above 0.5, 1 and 2.

### 2.1 What goes in

The forced responses of hist-nat and `EXPERIMENT` and the signal between them, then their noise and the pooled noise; `MODEL`.

In [ ]:
#(t): The year shown: the centre of the final WINDOW years
sn_year = int(lesfmip_seasonal_sn_ds['year'][LAST_FULL_YEAR])
sn_year

In [ ]:
#(t): hist-nat's forced response in that year, MODEL, every season
hist_nat_forced_da = lesfmip_forced_ds[VARIABLE].sel(model=MODEL, experiment='hist-nat', season=SEASONS, year=sn_year)
hist_nat_forced_da

In [ ]:
#(t): EXPERIMENT's
experiment_forced_da = lesfmip_forced_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS, year=sn_year)
experiment_forced_da

In [ ]:
#(t): The signal (lesfmip_signal_ds holds the same)
signal_da = experiment_forced_da - hist_nat_forced_da
signal_da

In [ ]:
panels = fr.row_grid(
    [fr.Row(hist_nat_forced_da, scales.temperature, 'hist-nat\nforced response'),
     fr.Row(experiment_forced_da, scales.temperature, f'{EXPERIMENT}\nforced response'),
     fr.Row(signal_da, scales.change, f'Signal\n{EXPERIMENT} − hist-nat')],
    title=f'What goes into the signal: {MODEL}, {EXPERIMENT}, {sn_year}',
)

In [ ]:
#(t): hist-nat's standard deviation about its forced response, in the WINDOW years centred on sn_year
hist_nat_sigma_da = lesfmip_experiment_noise_ds[VARIABLE].sel(model=MODEL, experiment='hist-nat', season=SEASONS,
                                                              year=sn_year)
hist_nat_sigma_da

In [ ]:
#(t): EXPERIMENT's
experiment_sigma_da = lesfmip_experiment_noise_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS,
                                                                year=sn_year)
experiment_sigma_da

In [ ]:
#(t): The noise: the two pooled (lesfmip_noise_ds holds the same)
noise_da = np.sqrt(0.5 * (hist_nat_sigma_da ** 2 + experiment_sigma_da ** 2))
noise_da

In [ ]:
panels = fr.row_grid(
    [fr.Row(hist_nat_sigma_da, scales.noise, 'hist-nat\nσ'),
     fr.Row(experiment_sigma_da, scales.noise, f'{EXPERIMENT}\nσ'),
     fr.Row(noise_da, scales.noise, 'Noise\nthe two pooled')],
    title=f'What goes into the noise: {MODEL}, {EXPERIMENT}, the {WINDOW} years centred on {sn_year}',
)

### 2.2 One model

The signal, the noise and |S/N| for every experiment of `MODEL`.

In [ ]:
#(t): MODEL's signal in every experiment and season, sn_year
model_signal_da = lesfmip_signal_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS, year=sn_year)
model_signal_da

In [ ]:
#(t): Its noise
model_noise_da = lesfmip_noise_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS, year=sn_year)
model_noise_da

In [ ]:
#(t): S/N
model_sn_da = lesfmip_seasonal_sn_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS, year=sn_year)
model_sn_da

In [ ]:
#(t): |S/N|: how far apart the distributions have moved, whichever way
model_abs_sn_da = np.abs(model_sn_da)

In [ ]:
panels = fr.map_grid(model_signal_da, scales.change, title=f'{MODEL}: signal, {sn_year}',
                     row_dim='experiment', col_dim='season')

In [ ]:
panels = fr.map_grid(model_noise_da, scales.noise, title=f'{MODEL}: noise, {sn_year}',
                     row_dim='experiment', col_dim='season')

In [ ]:
panels = fr.map_grid(model_abs_sn_da, scales.sn, title=f'{MODEL}: |S/N|, {sn_year}',
                     row_dim='experiment', col_dim='season')

### 2.3 Every model

One figure per experiment: every model down, every season across.

In [ ]:
#(t): S/N for every model, experiment and season, sn_year
all_sn_da = lesfmip_seasonal_sn_ds[VARIABLE].sel(experiment=EXPERIMENTS, season=SEASONS, year=sn_year)
all_sn_da

In [ ]:
#(t): |S/N|
all_abs_sn_da = np.abs(all_sn_da)

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_abs_sn_da.sel(experiment=experiment), scales.sn, col_dim='season',
        title=f'|S/N|, every model: {experiment}, {sn_year}',
    )

### 2.4 How many models

The multi-model median of |S/N| (top row), then the number of models with |S/N| above 0.5, 1 and 2. Grey is up to half the models, green more than half. One figure per experiment.

In [ ]:
#(t): The levels, on a 'level' dim, so every model is compared with each at once
sn_levels_da = xr.DataArray(SN_LEVELS, dims='level', coords={'level': SN_LEVELS})
sn_levels_da

In [ ]:
#(t): How many models have |S/N| above each level
sn_counts_da = (all_abs_sn_da > sn_levels_da).sum('model')
sn_counts_da

In [ ]:
#(t): The multi-model median of |S/N|
median_abs_sn_da = all_abs_sn_da.median('model')
median_abs_sn_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.count_grid(
        sn_counts_da.sel(experiment=experiment), N_MODELS,
        above=[fr.Row(median_abs_sn_da.sel(experiment=experiment), scales.sn, 'Multi-model\nmedian')],
        title=f'How many models have moved how far: |S/N| of {experiment}, {sn_year}',
        label='Number of models with |S/N| above the level',
        row_dim='level', col_dim='season', signed=False,
        row_labels={level: f'|S/N| > {level:g}' for level in SN_LEVELS},
    )